# Banking Operations Intelligence

# SQL Business Analysis

## Business Context

SQL is one of the most widely used languages for querying enterprise databases in financial institutions. This notebook demonstrates how SQL can be used to answer operational and business questions using banking transaction data.

The dataset was loaded into an SQLite database, and SQL queries were executed to simulate real-world analytical workflows commonly used by banking technology and analytics teams.

In [1]:
import pandas as pd
import sqlite3

In [3]:
from google.colab import files

uploaded = files.upload()

Saving bank_transactions_data_2.csv to bank_transactions_data_2.csv


In [4]:
conn = sqlite3.connect("banking_operations.db")

In [6]:
df = pd.read_csv("bank_transactions_data_2.csv")

df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [7]:
df.to_sql(
    "bank_transactions",
    conn,
    if_exists="replace",
    index=False
)

2512

In [8]:
query = """
SELECT *
FROM bank_transactions
LIMIT 5;
"""

pd.read_sql(query, conn)

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


# Executive Overview

The following SQL queries provide high-level business metrics describing the bank's transaction ecosystem.

## Request from Chief Operations Officer

Provide an executive summary of banking transactions, customer activity, and transaction value across the organization.

In [9]:
query = """

SELECT

COUNT(*) AS total_transactions,

COUNT(DISTINCT AccountID) AS unique_customers,

ROUND(SUM(TransactionAmount),2) AS total_transaction_value,

ROUND(AVG(TransactionAmount),2) AS average_transaction_value

FROM bank_transactions;

"""

summary = pd.read_sql(query, conn)

summary

,total_transactions,unique_customers,total_transaction_value,average_transaction_value
0,2512,495,747555.57,297.59


### Business Interpretation

This query provides an executive snapshot of the bank's transaction activity, customer base, and transaction values.

## Request from Head of Digital Banking

Compare transaction volumes across banking channels to understand customer adoption of digital and physical banking services.

In [10]:
query = """

SELECT

Channel,

COUNT(*) AS total_transactions

FROM bank_transactions

GROUP BY Channel

ORDER BY total_transactions DESC;

"""

channel = pd.read_sql(query, conn)

channel

,Channel,total_transactions
0,Branch,868
1,ATM,833
2,Online,811


### Business Interpretation

Understanding channel utilization helps banks optimize digital banking investments and branch operations.

# Customer Analytics

These queries analyze customer demographics and transaction behaviour.

## Request from Customer Strategy Team

Identify which customer occupations generate the highest average transaction value to support customer segmentation initiatives.

In [13]:
query = """

SELECT

CustomerOccupation,

COUNT(*) AS total_transactions,

ROUND(AVG(TransactionAmount),2) AS average_transaction

FROM bank_transactions

GROUP BY CustomerOccupation

ORDER BY average_transaction DESC;

"""

occupation = pd.read_sql(query, conn)

occupation

,CustomerOccupation,total_transactions,average_transaction
0,Student,657,313.22
1,Retired,599,294.53
2,Doctor,631,292.70
3,Engineer,625,289.04


## Request from Customer Insights Team

Analyze transaction activity across different age groups to better understand customer demographics.

In [12]:
query = """

SELECT

CASE

WHEN CustomerAge <25 THEN '18-24'

WHEN CustomerAge <35 THEN '25-34'

WHEN CustomerAge <45 THEN '35-44'

WHEN CustomerAge <55 THEN '45-54'

ELSE '55+'

END AS age_group,

COUNT(*) AS total_transactions,

ROUND(AVG(TransactionAmount),2) AS average_transaction

FROM bank_transactions

GROUP BY age_group

ORDER BY total_transactions DESC;

"""

age = pd.read_sql(query, conn)

age

,age_group,total_transactions,average_transaction
0,55+,868,299.32
1,25-34,534,310.58
2,45-54,406,268.47
3,18-24,402,315.88
4,35-44,302,284.48


## Request from Regional Operations Team

Identify the locations with the highest transaction volumes to support operational planning and resource allocation.

In [11]:
query = """

SELECT

Location,

COUNT(*) AS total_transactions

FROM bank_transactions

GROUP BY Location

ORDER BY total_transactions DESC

LIMIT 10;

"""

locations = pd.read_sql(query, conn)

locations

,Location,total_transactions
0,Fort Worth,70
1,Los Angeles,69
2,Oklahoma City,68
3,Charlotte,68
4,Tucson,67
5,Philadelphia,67
6,Omaha,65
7,Miami,64
8,Memphis,63
9,Houston,63


# Operations Analytics

The following queries evaluate operational efficiency across banking channels and identify patterns that can support process optimization.

## Request from Banking Operations Team

Evaluate transaction processing times across banking channels to identify operational bottlenecks.

In [19]:
query = """

SELECT

Channel,

ROUND(AVG(TransactionDuration),2) AS average_processing_time_seconds

FROM bank_transactions

GROUP BY Channel

ORDER BY average_processing_time_seconds DESC;

"""

processing_time = pd.read_sql(query, conn)

processing_time

,Channel,average_processing_time_seconds
0,ATM,122.09
1,Online,120.31
2,Branch,116.68


### Business Interpretation

Comparing average transaction duration across channels helps identify operational bottlenecks and opportunities to improve customer experience.

## Request from Merchant Services Team

Identify merchants contributing the highest transaction value to understand business concentration.

In [18]:
query = """

SELECT

MerchantID,

COUNT(*) AS transaction_count,

ROUND(SUM(TransactionAmount),2) AS total_transaction_value

FROM bank_transactions

GROUP BY MerchantID

ORDER BY total_transaction_value DESC

LIMIT 10;

"""

merchant = pd.read_sql(query, conn)

merchant

,MerchantID,transaction_count,total_transaction_value
0,M026,45,13865.15
1,M066,34,11948.75
2,M005,32,11099.93
3,M048,26,10954.79
4,M009,30,10526.04
5,M038,31,10443.51
6,M013,33,10416.35
7,M028,33,10316.32
8,M081,23,10154.77
9,M018,27,10117.83


### Business Interpretation

Merchant-level analysis helps identify high-volume business partners and understand transaction concentration across the merchant network.

## Request from Customer Engagement Team

Identify the most active customer accounts based on transaction frequency.

In [17]:
query = """

SELECT

AccountID,

COUNT(*) AS total_transactions,

ROUND(SUM(TransactionAmount),2) AS total_transaction_value

FROM bank_transactions

GROUP BY AccountID

ORDER BY total_transactions DESC

LIMIT 10;

"""

accounts = pd.read_sql(query, conn)

accounts

,AccountID,total_transactions,total_transaction_value
0,AC00460,12,5570.34
1,AC00363,12,4702.91
2,AC00362,12,2991.32
3,AC00202,12,3722.12
4,AC00480,11,3309.50
5,AC00456,11,3090.60
6,AC00304,11,3651.80
7,AC00257,11,3410.02
8,AC00225,11,2926.30
9,AC00297,10,2733.73


### Business Interpretation

Frequently active accounts may represent highly engaged customers or accounts that require additional operational monitoring depending on transaction patterns.

# Risk Analytics

Operational risk monitoring is an important component of banking analytics. The following queries identify transaction characteristics that may require additional review.

## Request from Risk Analytics Team

Identify transactions involving multiple login attempts for operational risk monitoring.

In [16]:
query = """

SELECT

TransactionID,

AccountID,

LoginAttempts,

TransactionAmount,

Channel

FROM bank_transactions

WHERE LoginAttempts >= 3

ORDER BY LoginAttempts DESC;

"""

risk = pd.read_sql(query, conn)

risk.head(15)

,TransactionID,AccountID,LoginAttempts,TransactionAmount,Channel
0,TX000027,AC00441,5,246.93,ATM
1,TX000148,AC00161,5,514.95,Online
2,TX000275,AC00454,5,1176.28,ATM
3,TX000395,AC00326,5,6.30,Branch
4,TX000415,AC00495,5,83.50,Branch
5,TX000464,AC00417,5,302.16,Online
6,TX000492,AC00318,5,505.19,ATM
7,TX000509,AC00353,5,127.00,Branch
8,TX000686,AC00071,5,119.30,Branch
9,TX000692,AC00418,5,25.94,Online


### Business Interpretation

Multiple login attempts may indicate authentication challenges or unusual customer behavior. These transactions can be prioritized for operational review.

## Request from Wealth Management Team

Analyze average account balances across customer occupations to understand customer value segments.

In [15]:
query = """

SELECT

CustomerOccupation,

ROUND(AVG(AccountBalance),2) AS average_balance

FROM bank_transactions

GROUP BY CustomerOccupation

ORDER BY average_balance DESC;

"""

balance = pd.read_sql(query, conn)

balance

,CustomerOccupation,average_balance
0,Doctor,8978.99
1,Engineer,5486.41
2,Retired,4542.16
3,Student,1570.21


### Business Interpretation

Understanding average account balances across customer segments supports customer profiling and the development of tailored financial products.

## Request from Retail Banking Team

Compare debit and credit transaction patterns to understand customer transaction behavior.

In [14]:
query = """

SELECT

TransactionType,

COUNT(*) AS total_transactions,

ROUND(SUM(TransactionAmount),2) AS total_transaction_value,

ROUND(AVG(TransactionAmount),2) AS average_transaction_value

FROM bank_transactions

GROUP BY TransactionType;

"""

transaction_type = pd.read_sql(query, conn)

transaction_type

,TransactionType,total_transactions,total_transaction_value,average_transaction_value
0,Credit,568,174092.57,306.50
1,Debit,1944,573463.00,294.99


### Business Interpretation

Comparing debit and credit transaction patterns provides insight into customer spending and account usage behavior.

# Executive Summary

## Key Findings

- Banking channels demonstrate different transaction volumes and processing times, highlighting opportunities for operational optimization.
- Customer demographics, including occupation and age, influence transaction behavior and average account balances.
- Merchant-level analysis reveals transaction concentration across key merchants.
- Transactions involving multiple login attempts can be monitored as operational risk indicators.
- SQL enables efficient analysis of structured banking data and complements Python-based exploratory analysis.

---

## Business Impact

This SQL analysis demonstrates how structured query language can be used to answer business questions commonly encountered by banking operations, technology, and analytics teams. The findings support data-driven decision-making in areas such as customer segmentation, operational efficiency, and transaction monitoring.